# ARGOS Bot — Ensemble Training (Colab GPU)

Train LSTM + XGBoost + MetaModel + Calibrator for multiple symbols.

## Workflow
1. Mount Google Drive (folder: `argos_training_work`)
2. Place `argos_training_data_15m_tb.zip` in that folder
3. This notebook trains everything with GPU
4. Download the resulting `checkpoints.zip` and import via `python -m scripts.import_checkpoint --input checkpoints.zip`

**Dataset**: 6 symbols (BTC, ETH, SOL, XRP, DOGE, AVAX), 15m, 4 years, Triple Barrier labels.
**Requires**: TensorFlow, XGBoost, scikit-learn, pandas, numpy, pyarrow

In [ ]:
# Install deps (run once)
!pip install tensorflow xgboost scikit-learn pandas numpy pyarrow -q

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

In [ ]:
import hashlib
import json
import os
import shutil
import tempfile
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

print('TensorFlow:', tf.__version__)
print('GPU:', 'YES' if tf.config.list_physical_devices('GPU') else 'NO')

# Check GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
# ── Setup ──────────────────────────────────────────────────────

# Paths
DATA_ZIP = 'argos_training_data_15m_tb.zip'  # Triple Barrier dataset
OUTPUT_ZIP = 'checkpoints.zip'
WORK_DIR = Path('/content/drive/MyDrive/argos_training_work')
CHECKPOINT_DIR = WORK_DIR / 'checkpoints'

# LSTM params (from ModelConfig defaults)
LOOKBACK = 20
TARGET_LOOKAHEAD = 5  # Triple Barrier max_holding
BATCH_SIZE = 32
MAX_EPOCHS = 20
DROPOUT = 0.2
EARLY_STOP_PATIENCE = 4
CONFIDENCE_THRESHOLD = 0.7

# Walk Forward
N_WINDOWS = 5
WINDOW_VAL_PCT = 0.15

WORK_DIR.mkdir(parents=True, exist_ok=True)

## 1. Extract data

In [ ]:
if not (WORK_DIR / 'manifest.json').exists():
    print('Extracting dataset...')
    with zipfile.ZipFile(WORK_DIR / DATA_ZIP, 'r') as zf:
        zf.extractall(str(WORK_DIR))
else:
    print('manifest.json found — skipping extraction')

with open(WORK_DIR / 'manifest.json') as f:
    manifest = json.load(f)

print('Manifest:', json.dumps(manifest, indent=2)[:500])

In [ ]:
symbols = manifest['symbols']
print(f'Loaded {len(symbols)} symbols:')
for s in symbols:
    print(f"  {s['symbol']}: {s['n_samples']} samples, {s['n_features']} features")

## 2. Define building blocks

In [ ]:
def build_lstm(n_features: int) -> tf.keras.Model:
    """Build LSTM architecture matching NovaQuantKerasModel."""
    inputs = tf.keras.Input(shape=(LOOKBACK, n_features), name='ohlcv_window')
    x = tf.keras.layers.LSTM(128, return_sequences=True, name='lstm_0')(inputs)
    x = tf.keras.layers.Dropout(DROPOUT, name='dropout_0')(x)
    x = tf.keras.layers.LSTM(64, return_sequences=True, name='lstm_1')(x)
    x = tf.keras.layers.Dropout(DROPOUT, name='dropout_1')(x)
    x = tf.keras.layers.LSTM(32, name='lstm_2')(x)
    x = tf.keras.layers.Dropout(DROPOUT, name='dropout_2')(x)
    x = tf.keras.layers.Dense(16, activation='relu', name='dense')(x)
    x = tf.keras.layers.Dropout(DROPOUT, name='dropout_dense')(x)
    outputs = tf.keras.layers.Dense(3, activation='softmax', name='output')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name='NovaQuant')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


def make_windows(features: np.ndarray) -> np.ndarray:
    """Sliding window view: (n, lookback, n_features)."""
    windows = np.lib.stride_tricks.sliding_window_view(features, LOOKBACK, axis=0)
    return windows.transpose(0, 2, 1)

In [ ]:
def train_lstm(
    x_train: np.ndarray, y_train: np.ndarray,
    x_val: np.ndarray, y_val: np.ndarray,
    n_features: int,
) -> tuple[tf.keras.Model, dict]:
    """Train LSTM with early stopping."""
    model = build_lstm(n_features)

    cb = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=EARLY_STOP_PATIENCE,
            restore_best_weights=True, verbose=0,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=EARLY_STOP_PATIENCE // 2, min_lr=1e-6, verbose=0,
        ),
    ]

    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
        callbacks=cb, verbose=1,
    )

    val_loss = float(min(history.history['val_loss']))
    best_idx = history.history['val_loss'].index(val_loss)
    val_acc = float(history.history['val_accuracy'][best_idx])

    metrics = {
        'val_loss': val_loss,
        'val_accuracy': val_acc,
        'epochs_trained': len(history.history['loss']),
    }
    return model, metrics

In [ ]:
def train_xgboost(
    x_train: np.ndarray, y_train: np.ndarray,
    x_val: np.ndarray, y_val: np.ndarray,
) -> tuple['xgb.XGBClassifier', dict]:
    """Train XGBoost classifier."""
    n = x_train.shape[0] * x_train.shape[1]
    x_train_2d = x_train.reshape(x_train.shape[0], -1)
    x_val_2d = x_val.reshape(x_val.shape[0], -1)
    y_train_lab = np.argmax(y_train, axis=1)
    y_val_lab = np.argmax(y_val, axis=1)

    model = xgb.XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        objective='multi:softprob', num_class=3,
        eval_metric='mlogloss', early_stopping_rounds=10,
        verbosity=0,
    )
    model.fit(
        x_train_2d, y_train_lab,
        eval_set=[(x_val_2d, y_val_lab)],
        verbose=False,
    )

    val_pred = model.predict_proba(x_val_2d)
    val_acc = float(np.mean(np.argmax(val_pred, axis=1) == y_val_lab))

    return model, {'val_accuracy': val_acc}

In [ ]:
def predict_lstm(model: tf.keras.Model, x: np.ndarray) -> tuple[float, float, float]:
    """LSTM forward pass -> (buy, sell, hold) probs."""
    batch = np.expand_dims(x, axis=0)
    probs = model.predict(batch, verbose=0)[0]
    return (float(probs[0]), float(probs[1]), float(probs[2]))


def predict_xgb(model: 'xgb.XGBClassifier', x: np.ndarray) -> tuple[float, float, float]:
    """XGBoost forward pass -> (buy, sell, hold) probs."""
    flat = x.reshape(1, -1)
    probs = model.predict_proba(flat)[0]
    return (float(probs[0]), float(probs[1]), float(probs[2]))


def signal_to_probs(probs: tuple[float, float, float]) -> tuple[float, float, float]:
    """Normalise to sum=1."""
    total = sum(probs)
    if total > 0:
        return tuple(p / total for p in probs)
    return (1/3, 1/3, 1/3)

## 3. Train per symbol

In [ ]:
import io
import joblib
import time

results = []

# ── Resume: skip symbols with existing checkpoints ──────
completed = set()
for sym_info in symbols:
    key = sym_info['key']
    sym_dir = CHECKPOINT_DIR / key
    if sym_dir.exists() and any(sym_dir.iterdir()):
        completed.add(key)

if completed:
    print(f'Resuming: {len(completed)} symbols ready ({', '.join(sorted(completed))})')
else:
    print('Fresh start — no previous checkpoints found')

for sym_info in symbols:
    key = sym_info['key']
    symbol = sym_info['symbol']

    # ── Skip if already trained ──────────────────────────
    if key in completed:
        print(f'\n  ⏭️  {symbol} already trained — skipping')
        continue

    print(f'\n{"="*60}')
    print(f'[{symbol}] Starting training')
    print(f'{"="*60}')
    t0 = time.time()

    # ── [1/4] Load features ──────────────────────────────
    print(f'  [1/4] Loading features...')
    feat_df = pd.read_parquet(WORK_DIR / f'features/{key}_features.parquet')
    tgt_df = pd.read_parquet(WORK_DIR / f'features/{key}_targets.parquet')
    features_raw = feat_df.values.astype(np.float32)
    targets = tgt_df.values.astype(np.float32)
    n_features = features_raw.shape[1]
    print(f'  [1/4] Loaded {len(feat_df)} samples, {n_features} features')
    label_dist = targets.sum(axis=0)
    print(f'         Labels: BUY={label_dist[0]/len(targets)*100:.1f}% SELL={label_dist[1]/len(targets)*100:.1f}% HOLD={label_dist[2]/len(targets)*100:.1f}%')

    # Map column names to indices dynamically
    cols = feat_df.columns
    bb_upper_idx = cols.get_loc('bb_upper')
    bb_middle_idx = cols.get_loc('bb_middle')
    bb_lower_idx = cols.get_loc('bb_lower')
    adx_idx = cols.get_loc('adx')
    atr_idx = cols.get_loc('atr')
    rsi_idx = cols.get_loc('rsi')
    volume_idx = cols.get_loc('volume')

    # ── [2/4] Create windows ─────────────────────────────
    print(f'  [2/4] Creating windows (lookback={LOOKBACK})...')
    windows = make_windows(features_raw)
    aligned = targets[LOOKBACK - 1:]
    aligned = aligned[:len(windows)]
    print(f'  [2/4] {len(windows)} windows')

    n = len(windows)
    window_size = n // N_WINDOWS

    all_meta_features = []
    all_oof_targets = []
    lstm_models = []
    xgb_models = []

    # ── [3/4] Walk Forward ────────────────────────────────
    for w in range(N_WINDOWS):
        start = w * window_size
        end = n if w == N_WINDOWS - 1 else (w + 1) * window_size
        print(f'\n  Window {w+1}/{N_WINDOWS} (samples {start}-{end})')

        w_data_raw = windows[start:end]
        w_tgts = aligned[start:end]
        n_total = len(w_data_raw)
        n_val = max(1, int(n_total * WINDOW_VAL_PCT))
        n_train = n_total - n_val

        # Split before normalising
        x_tr_raw = w_data_raw[:n_train]
        x_va_raw = w_data_raw[n_train:]
        y_tr = w_tgts[:n_train]
        y_va = w_tgts[n_train:]

        # Per-window z-score: compute from train ONLY
        w_means = np.mean(x_tr_raw, axis=(0, 1))
        w_stds = np.std(x_tr_raw, axis=(0, 1)) + 1e-6

        x_tr = (x_tr_raw - w_means) / w_stds
        x_va = (x_va_raw - w_means) / w_stds

        print(f'    Train: {len(x_tr)} | Val: {len(x_va)}')

        # LSTM
        print(f'    Training LSTM...')
        lstm, lstm_metrics = train_lstm(x_tr, y_tr, x_va, y_va, n_features)
        print(f'    LSTM done — val_acc: {lstm_metrics["val_accuracy"]:.4f}')

        # XGBoost
        print(f'    Training XGBoost...')
        xgb_model, xgb_metrics = train_xgboost(x_tr, y_tr, x_va, y_va)
        print(f'    XGBoost done — val_acc: {xgb_metrics["val_accuracy"]:.4f}')

        # OOF predictions on validation split
        print(f'    Computing OOF predictions...')
        oof_lstm = np.array([
            signal_to_probs(predict_lstm(lstm, x_va[i]))
            for i in range(len(x_va))
        ])
        oof_xgb = np.array([
            signal_to_probs(predict_xgb(xgb_model, x_va[i]))
            for i in range(len(x_va))
        ])

        # Meta features with dynamic indices + real BBW
        for i in range(len(x_va)):
            global_idx = start + n_train + i
            raw_row = features_raw[global_idx]
            bbw = float((raw_row[bb_upper_idx] - raw_row[bb_lower_idx]) /
                        (raw_row[bb_middle_idx] + 1e-10))
            adx_val = float(raw_row[adx_idx]) if n_features > adx_idx else 25.0
            atr_val = float(raw_row[atr_idx]) if n_features > atr_idx else 100.0
            rsi_val = float(raw_row[rsi_idx]) if n_features > rsi_idx else 50.0
            vol_val = float(raw_row[volume_idx]) if n_features > volume_idx else 1000.0
            mf = [*oof_lstm[i], *oof_xgb[i], adx_val, bbw, atr_val, rsi_val, vol_val]
            all_meta_features.append(mf)
            all_oof_targets.append(y_va[i])

        lstm_models.append(lstm)
        xgb_models.append(xgb_model)

        # Save last window normalisation stats for inference
        last_w_means = w_means.copy()
        last_w_stds = w_stds.copy()

    # ── [4/4] MetaModel + Calibrator ───────────────────────
    print(f'\n  [4/4] MetaModel + Calibrator...')
    meta_features = np.array(all_meta_features)
    meta_targets = np.array(all_oof_targets)
    meta_labels = np.argmax(meta_targets, axis=1)
    print(f'  [4/4] MetaFeatures: {meta_features.shape}')

    # MetaModel (XGBoost stacking on OOF predictions)
    meta_model = xgb.XGBClassifier(
        n_estimators=50, max_depth=3, learning_rate=0.1,
        objective='multi:softprob', num_class=3,
        eval_metric='mlogloss', verbosity=0,
    )
    meta_model.fit(meta_features, meta_labels)
    meta_pred = meta_model.predict_proba(meta_features)
    meta_acc = float(np.mean(np.argmax(meta_pred, axis=1) == meta_labels))
    print(f'  [4/4] MetaModel train_acc: {meta_acc:.4f}')

    # Calibrator (Platt Scaling on MetaModel output)
    max_probs = np.max(meta_pred, axis=1)
    print(f'  [4/4] Max probs range: [{max_probs.min():.4f}, {max_probs.max():.4f}]')
    calibrator = LogisticRegression(C=1.0, solver='lbfgs')
    calibrator.fit(max_probs.reshape(-1, 1), (meta_labels == 0).astype(int))
    print(f'  [4/4] Calibrator trained (BUY label count: {(meta_labels == 0).sum()}/{len(meta_labels)})')

    # ── Save final model ───────────────────────────────────
    final_lstm = lstm_models[-1]
    final_xgb = xgb_models[-1]

    version = f'v1.0.{int(datetime.now(timezone.utc).timestamp())}'
    version_dir = CHECKPOINT_DIR / key / version
    version_dir.mkdir(parents=True, exist_ok=True)

    # Serialise LSTM weights (Keras 3 format)
    print(f'  Saving to {version_dir}...')
    weights_path = version_dir / 'lstm_weights.keras'
    final_lstm.save(weights_path)
    weights_bytes = weights_path.read_bytes()
    weights_hash = hashlib.sha256(weights_bytes).hexdigest()[:16]
    shutil.copy(weights_path, version_dir / 'weights.keras')

    # model_config.json
    config = {
        'lookback': LOOKBACK,
        'confidence_threshold': CONFIDENCE_THRESHOLD,
        'layers': [128, 64, 32, 16],
        'features': list(feat_df.columns),
        'target_lookahead': TARGET_LOOKAHEAD,
        'target_return_pct': 1.0,  # N/A for Triple Barrier
        'dropout_rate': DROPOUT,
        'batch_size': BATCH_SIZE,
        'max_epochs': MAX_EPOCHS,
        'early_stop_patience': EARLY_STOP_PATIENCE,
        'label_method': 'triple_barrier',
        'atr_multiplier': 1.5,
    }
    with open(version_dir / 'model_config.json', 'w') as f:
        json.dump(config, f, indent=2)

    # model_metadata.json
    metadata = {
        'model_version': version,
        'trained_at': datetime.now(timezone.utc).isoformat(),
        'weights_hash': weights_hash,
        'feature_means': [float(m) for m in last_w_means],
        'feature_stds': [float(s) for s in last_w_stds],
        'metrics': {
            'lstm_val_accuracy': lstm_metrics['val_accuracy'],
            'xgb_val_accuracy': xgb_metrics['val_accuracy'],
            'meta_train_accuracy': meta_acc,
        },
    }
    with open(version_dir / 'model_metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)

    # Extra: xgboost model, meta model, calibrator
    final_xgb.save_model(str(version_dir / 'xgboost_model.json'))
    meta_model.save_model(str(version_dir / 'meta_model.json'))
    joblib.dump(calibrator, version_dir / 'calibrator.pkl')

    results.append({
        'symbol': symbol,
        'key': key,
        'version': version,
        'lstm_acc': lstm_metrics['val_accuracy'],
        'xgb_acc': xgb_metrics['val_accuracy'],
        'meta_acc': meta_acc,
        'n_samples': len(all_oof_targets),
    })

    elapsed = time.time() - t0
    print(f'  ✅ {symbol} → {version} ({elapsed:.1f}s)')
    print(f'     LSTM: {lstm_metrics["val_accuracy"]:.4f} | XGB: {xgb_metrics["val_accuracy"]:.4f} | Meta: {meta_acc:.4f}')

print('\n' + '='*60)
print('Training complete!')
print(f'Total symbols: {len(results)}')
print('='*60)

## 4. Generate predictions.parquet

In [ ]:
# ── Generate predictions.parquet for all trained symbols ──
print('Generating predictions.parquet for all symbols...')
print()

for sym_info in symbols:
    key = sym_info['key']
    symbol = sym_info['symbol']
    sym_dir = CHECKPOINT_DIR / key

    if not (sym_dir.exists() and any(sym_dir.iterdir())):
        print(f'  ⏭️  {symbol} — no checkpoint, skipping')
        continue

    versions = sorted(sym_dir.iterdir())
    latest = versions[-1]
    print(f'\n  ── {symbol} ({latest.name}) ──')

    print(f'  [1/7] Loading features...')
    feat_df = pd.read_parquet(WORK_DIR / f'features/{key}_features.parquet')
    tgt_df = pd.read_parquet(WORK_DIR / f'features/{key}_targets.parquet')

    if 'adx' not in feat_df.columns:
        print(f'  [1b] ADX missing — computing from OHLCV...')
        import ta
        adx_ind = ta.trend.ADXIndicator(
            high=feat_df['high'], low=feat_df['low'],
            close=feat_df['close'], window=14,
        )
        feat_df['adx'] = adx_ind.adx()

    print(f'  [2/7] Loading checkpoint...')
    meta_model = xgb.XGBClassifier()
    meta_model.load_model(str(latest / 'meta_model.json'))
    calibrator = joblib.load(latest / 'calibrator.pkl')
    with open(latest / 'model_metadata.json') as f:
        metadata = json.load(f)
    with open(latest / 'model_config.json') as f:
        cfg = json.load(f)
    means = np.array(metadata['feature_means'], dtype=np.float64)
    stds = np.array(metadata['feature_stds'], dtype=np.float64) + 1e-6
    lookback = cfg['lookback']
    print(f'  [2] lookback={lookback}')

    print(f'  [3/7] Loading LSTM + XGBoost...')
    final_lstm = tf.keras.models.load_model(str(latest / 'lstm_weights.keras'))
    final_xgb = xgb.XGBClassifier()
    final_xgb.load_model(str(latest / 'xgboost_model.json'))

    cols = list(feat_df.columns)
    bb_upper_idx = cols.index('bb_upper')
    bb_middle_idx = cols.index('bb_middle')
    bb_lower_idx = cols.index('bb_lower')
    adx_idx = cols.index('adx')
    atr_idx = cols.index('atr')
    rsi_idx = cols.index('rsi')
    volume_idx = cols.index('volume')

    print(f'  [4/7] Preparing test set...')
    test_size = max(20000, int(len(feat_df) * 0.2))
    test_df = feat_df.iloc[-test_size:]
    test_targets = tgt_df.iloc[-test_size:].values
    test_norm = (test_df.values.astype(np.float64) - means) / stds
    test_windows = make_windows(test_norm)
    n_val = len(test_windows)
    aligned_targets = test_targets[lookback - 1:]
    aligned_targets = aligned_targets[:n_val]
    print(f'  [4] Test: {test_size} samples → {n_val} windows')

    print(f'  [5/7] Running inference...')
    rows = []
    for i in range(0, n_val, BATCH_SIZE):
        batch_end = min(i + BATCH_SIZE, n_val)
        batch = test_windows[i:batch_end]
        bsize = batch_end - i

        lstm_probs = final_lstm.predict(batch, verbose=0)
        xgb_probs = final_xgb.predict_proba(batch.reshape(bsize, -1))

        for j in range(bsize):
            idx = i + j
            raw_row = test_df.values[idx]
            bbw = float((raw_row[bb_upper_idx] - raw_row[bb_lower_idx]) /
                        (raw_row[bb_middle_idx] + 1e-10))
            mf = np.array([[
                *lstm_probs[j], *xgb_probs[j],
                float(raw_row[adx_idx]), bbw,
                float(raw_row[atr_idx]), float(raw_row[rsi_idx]),
                float(raw_row[volume_idx]),
            ]])
            meta_p = meta_model.predict_proba(mf)[0]
            max_prob = float(np.max(meta_p))
            calibrated = float(calibrator.predict_proba([[max_prob]])[0, 1])
            pred_class = int(np.argmax(meta_p))
            true_class = int(np.argmax(aligned_targets[idx]))

            rows.append({
                'pred_buy': float(meta_p[0]),
                'pred_sell': float(meta_p[1]),
                'pred_hold': float(meta_p[2]),
                'pred_class': pred_class,
                'true_class': true_class,
                'confidence': max_prob,
                'calibrated_buy': calibrated,
            })

        if (i // BATCH_SIZE) % 20 == 0:
            pct = int(100 * i / n_val)
            print(f'    Batch {i//BATCH_SIZE+1}/{(n_val+BATCH_SIZE-1)//BATCH_SIZE} ({pct}%)...')

    pred_df = pd.DataFrame(rows)
    pred_out = WORK_DIR / 'predictions' / f'{key}_predictions.parquet'
    pred_out.parent.mkdir(parents=True, exist_ok=True)
    pred_df.to_parquet(pred_out)
    print(f'  [6/7] ✅ Saved predictions ({len(pred_df)} rows)')

print(f'\n{"="*60}')
print('All predictions generated!')
print(f'{"="*60}')

In [ ]:
# Summary table
import pandas as pd
summary = pd.DataFrame(results)
print(summary.to_string(index=False))
print(f'\nSymbols with accuracy > 0.33: {(summary["meta_acc"] > 0.33).sum()}/{len(summary)}')

## 5. Export checkpoints


In [ ]:
print(f'Creating {OUTPUT_ZIP}...')

with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(CHECKPOINT_DIR):
        for fname in files:
            path = Path(root) / fname
            arcname = str(path.relative_to(CHECKPOINT_DIR.parent.parent))
            zf.write(path, arcname)

size_mb = os.path.getsize(OUTPUT_ZIP) / (1024 * 1024)
print(f'Created {OUTPUT_ZIP} ({size_mb:.1f} MB)')
print('\nDownload this file and import with:')
print('  python -m scripts.import_checkpoint --input checkpoints.zip')

In [ ]:
# Download trigger (Colab)
from google.colab import files
files.download(OUTPUT_ZIP)